In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Slider
from matplotlib.patches import FancyArrowPatch
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

BG      = "whitesmoke"
PANEL   = "white"
BORDER  = "silver"
TEXT    = "black"
MUTED   = "lightgrey"
CURRENT = "aquamarine"
ACCENT  = "gold"
VORTEX  = "darkviolet"   # optimal path in vortex flow
UNIFORM = "cyan"         # uniform-flow comparison

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   PANEL,
    "axes.edgecolor":   BORDER,
    "axes.labelcolor":  TEXT,
    "axes.titlecolor":  TEXT,
    "xtick.color":      MUTED,
    "ytick.color":      MUTED,
    "text.color":       TEXT,
    "grid.color":       BORDER,
    "grid.linewidth":   0.6,
    "font.family":      "sans-serif",
    "font.size":        9,
})

##flow field:  vx = -omega * y,  vy = omega * x  (rigid-body rotation)
#euler-Lagrange result: d(theta)/dt = omega  (heading rotates at vortex rate)

def ode_system_vortex(t, state, s, omega):
    x, y, theta = state
    x_dot     = s * np.cos(theta) - omega * y
    y_dot     = s * np.sin(theta) + omega * x
    theta_dot = omega                            
    return [x_dot, y_dot, theta_dot]


def integrate_path(theta0, s, omega, X=10.0, n_steps=500):

    def hit_target_x(t, state, s, omega):
        return state[0] - X
    hit_target_x.terminal  = True
    hit_target_x.direction = 1

    T_max = 4 * X / max(s * abs(np.cos(theta0)), 0.05)

    sol = solve_ivp(
        ode_system_vortex,
        [0, T_max],
        [0.0, 0.0, theta0],
        args=(s, omega),
        events=hit_target_x,
        max_step=T_max / n_steps,
        dense_output=True,
    )

    if sol.status == -1 or len(sol.t_events[0]) == 0:
        return None

    T  = sol.t_events[0][0]
    ts = np.linspace(0, T, n_steps)
    states = sol.sol(ts)
    return states[0], states[1], states[2], T


def y_at_arrival(theta0, s, omega, X=10.0):
    result = integrate_path(theta0, s, omega, X)
    if result is None:
        return np.nan
    _, ys, _, _ = result
    return ys[-1]


def shoot(s, omega, X=10.0):
    try:
        theta_min = -np.pi / 2 + 0.01
        theta_max =  np.pi / 2 - 0.01

        fa = y_at_arrival(theta_min, s, omega, X)
        fb = y_at_arrival(theta_max, s, omega, X)

        if np.isnan(fa) or np.isnan(fb) or fa * fb > 0:
            return None

        theta0 = brentq(y_at_arrival, theta_min, theta_max,
                        args=(s, omega, X), xtol=1e-6)

        result = integrate_path(theta0, s, omega, X)
        if result is None:
            return None
        xs, ys, thetas, T = result
        return theta0, xs, ys, thetas, T

    except Exception:
        return None


##visualisation

def draw_vortex_arrows(ax, omega, X=10.0, nx=6, ny=6):
    for coll in ax.collections[:]:
        coll.remove()
    for arrow in [c for c in ax.get_children()
                  if isinstance(c, FancyArrowPatch)]:
        arrow.remove()

    if abs(omega) < 1e-4:
        return

    xs = np.linspace(0.8, X - 0.8, nx)
    ys = np.linspace(-3.5, 3.5, ny)
    for x in xs:
        for y in ys:
            vx = -omega * y
            vy =  omega * x
            speed = np.sqrt(vx**2 + vy**2)
            if speed < 1e-4:
                continue
            scale = np.clip(speed * 0.2, 0.05, 0.6)
            ax.annotate(
                "", xy=(x + scale * vx / speed, y + scale * vy / speed),
                xytext=(x, y),
                arrowprops=dict(
                    arrowstyle="->",
                    color=CURRENT,
                    lw=0.7,
                    alpha=0.45,
                ),
            )


##main figure

def main():
    X      = 10.0
    s      = 2.0
    omega0 = 0.15

    fig = plt.figure(figsize=(13, 7), facecolor=BG)
    fig.suptitle(
        "Fish Migration — Vortex Flow  v = (−ωy, ωx)  (E-L + shooting method)",
        fontsize=11, color=TEXT, y=0.97, fontweight="bold",
    )

    gs = gridspec.GridSpec(2, 2, figure=fig,
                           left=0.07, right=0.97,
                           top=0.90, bottom=0.22,
                           hspace=0.40, wspace=0.32)

    ax_path  = fig.add_subplot(gs[:, 0])   # path plot (spans both rows)
    ax_theta = fig.add_subplot(gs[0, 1])   # heading angle along path
    ax_cost  = fig.add_subplot(gs[1, 1])   # travel time vs omega

    ##path axes
    ax_path.set_xlim(-0.5, X + 0.5)
    ax_path.set_ylim(-4.0, 4.0)
    ax_path.set_aspect("equal")
    ax_path.set_xlabel("x  (horizontal distance)", labelpad=3, fontsize=8)
    ax_path.set_ylabel("y  (lateral displacement)", labelpad=3, fontsize=8)
    ax_path.set_title("Optimal path in vortex flow", pad=5, fontsize=9)
    ax_path.grid(True, alpha=0.3)
    ax_path.axhline(0, color=BORDER, lw=1.0, zorder=1)
    ax_path.plot(0, 0, "o", color=TEXT,   ms=7,  zorder=5)
    ax_path.plot(X, 0, "*", color=VORTEX, ms=12, zorder=5)
    ax_path.text(-0.3, 0,   "Start",  color=MUTED,   fontsize=7,
                 va="center", ha="right")
    ax_path.text(X+0.2, 0, "Target", color=VORTEX, fontsize=7, va="center")

    vortex_line,  = ax_path.plot([], [], color=VORTEX,  lw=2.5,
                                  label="Vortex (E-L shooting)", zorder=4)
    straight_line,= ax_path.plot([], [], color=UNIFORM, lw=1.8,
                                  ls="--", label="No-flow straight path", zorder=3)
    ax_path.legend(loc="upper left", fontsize=7,
                   facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT,
                   borderpad=0.4, labelspacing=0.3)
    info_text = ax_path.text(
        0.98, 0.04, "", transform=ax_path.transAxes,
        fontsize=7, color=TEXT, va="bottom", ha="right", linespacing=1.4,
        bbox=dict(facecolor=PANEL, edgecolor=BORDER, boxstyle="round,pad=0.3"))

    ##theta axes
    ax_theta.set_xlabel("x  (position along river)", labelpad=2, fontsize=8)
    ax_theta.set_ylabel("θ  (degrees)", labelpad=2, fontsize=8)
    ax_theta.set_title("Heading angle along path", pad=4, fontsize=9)
    ax_theta.set_xlim(0, X)
    ax_theta.set_ylim(-95, 95)
    ax_theta.axhline(0, color=BORDER, lw=0.8, ls=":")
    ax_theta.grid(True, alpha=0.3)
    theta_line, = ax_theta.plot([], [], color=VORTEX, lw=2,
                                 label="Vortex (varies linearly)")
    ax_theta.legend(fontsize=7, facecolor=PANEL, edgecolor=BORDER,
                    labelcolor=TEXT, borderpad=0.4, labelspacing=0.3)

    ##travel time/omega axis
    ax_cost.set_xlabel("Vortex strength ω", labelpad=2, fontsize=8)
    ax_cost.set_ylabel("Travel time T", labelpad=2, fontsize=8)
    ax_cost.set_title("Travel time vs vortex strength", pad=4, fontsize=9)
    ax_cost.set_xlim(0, 0.45)
    ax_cost.grid(True, alpha=0.3)
    cost_line, = ax_cost.plot([], [], color=VORTEX, lw=2, label="Vortex")
    T_straight = X / s
    ax_cost.axhline(T_straight, color=UNIFORM, lw=1.5, ls="--",
                    label=f"No-flow T = {T_straight:.2f}")
    cost_dot,  = ax_cost.plot([], [], "o", color=ACCENT, ms=7, zorder=5)
    ax_cost.legend(fontsize=7, facecolor=PANEL, edgecolor=BORDER,
                   labelcolor=TEXT, borderpad=0.4, labelspacing=0.3)

    ##sliders
    ax_so = fig.add_axes([0.10, 0.13, 0.55, 0.022], facecolor=PANEL)
    ax_ss = fig.add_axes([0.10, 0.08, 0.55, 0.022], facecolor=PANEL)
    sl_omega = Slider(ax_so, "Vortex  ω", 0.0, 0.44, valinit=omega0, color=CURRENT)
    sl_s     = Slider(ax_ss, "Fish speed s", 0.5, 4.0, valinit=s,  color=VORTEX)
    for sl in (sl_omega, sl_s):
        sl.label.set_color(TEXT)
        sl.valtext.set_color(TEXT)

    ##updating
    def update(_=None):
        omega_val = sl_omega.val
        s_val     = sl_s.val

        draw_vortex_arrows(ax_path, omega_val, X)

        # straight-line reference (no flow)
        straight_line.set_data([0, X], [0, 0])
        T_ref = X / s_val

        result = shoot(s_val, omega_val, X)

        if result is None:
            vortex_line.set_data([], [])
            theta_line.set_data([], [])
            cost_dot.set_data([], [])
            info_text.set_text(
                f"ω={omega_val:.3f}  s={s_val:.2f}\n"
                f"No solution found")
        else:
            theta0, xs, ys, thetas, T = result
            vortex_line.set_data(xs, ys)
            theta_line.set_data(xs, np.degrees(thetas))
            cost_dot.set_data([min(omega_val, 0.44)], [T])
            info_text.set_text(
                f"ω={omega_val:.3f}  s={s_val:.2f}\n"
                f"θ(0) = {np.degrees(theta0):.1f}°\n"
                f"θ(T) = {np.degrees(thetas[-1]):.1f}°\n"
                f"T_vortex = {T:.3f}  T_straight = {T_ref:.3f}"
            )

        fig.canvas.draw_idle()

    def update_cost_curve(_=None):
        s_val     = sl_s.val
        omega_val = sl_omega.val
        T_ref     = X / s_val

        omegas = np.linspace(0.01, 0.44, 30)
        Ts = [shoot(s_val, om, X) for om in omegas]
        Ts = [r[4] if r is not None else np.nan for r in Ts]
        cost_line.set_data(omegas, Ts)
        ax_cost.lines[1].set_ydata([T_ref, T_ref])  # update straight-line ref

        valid = [t for t in Ts if not np.isnan(t)]
        if valid:
            ymax = max(max(valid), T_ref) * 1.1
            ax_cost.set_ylim(min(valid) * 0.9, ymax)

        fig.canvas.draw_idle()

    sl_omega.on_changed(update)
    sl_s.on_changed(update)

    def on_release(event):
        update_cost_curve()

    fig.canvas.mpl_connect("button_release_event", on_release)

    update()
    update_cost_curve()

    return sl_omega, sl_s


if __name__ == "__main__":
    sliders = main()
    plt.show()